# Organizing Metadata
Currently, our metadata is a bit scatter, but there are pointers to connect them all.
The purpose of this notebook is to explore how to best link all the metadata.

In [38]:
from pathlib import Path

import pandas as pd

# Download the CSV files by sheet from the gSheet metadata file based on name below
metadata_path_root = Path("../tmp/metadata")
image_metadata_df = pd.read_csv(metadata_path_root / "ImagePathMapping.csv")
preliminary_labels_df = pd.read_csv(metadata_path_root / "Main.csv")
catherine_labels_df = pd.read_csv(metadata_path_root / "Catherine_Labels.csv")
david_labels_df = pd.read_csv(metadata_path_root / "David_Labels.csv")

In [39]:
image_metadata_df.head()

,UUID,Original\nFilename,Site,Day,Plant #,Level,Index,View,Imaging\nSession #,Session\nFolder Name,Name of \nMapper,Recheck\nImage?,Notes
0,NaN,IMG_0861.png,5.0,21.0,NaN,NaN,1.0,D,4.0,session_4_1_21_2025,David,False,NaN
1,NaN,IMG_0862.png,5.0,21.0,NaN,NaN,1.0,V,4.0,session_4_1_21_2025,David,False,NaN
2,NaN,IMG_0865.png,5.0,21.0,NaN,NaN,2.0,D,4.0,session_4_1_21_2025,David,False,NaN
3,NaN,IMG_0866.png,5.0,21.0,NaN,NaN,2.0,V,4.0,session_4_1_21_2025,David,False,NaN
4,NaN,IMG_0871.png,5.0,21.0,NaN,NaN,3.0,D,4.0,session_4_1_21_2025,David,False,Seem to be missing ventral view from dataset


In [40]:
# Catherine labels and David labels should have the exact same structures
assert (david_labels_df.columns == catherine_labels_df.columns).all(), (
    "Structures are different"
)

labels_df = pd.concat([catherine_labels_df, david_labels_df])

assert labels_df.shape[0] == catherine_labels_df.shape[0] + david_labels_df.shape[0], (
    "Unclean merge"
)

labels_df.head()

,Leaf ID,Site,Day,Plant,Level,Index,Healthy,Leaf Miner?,Rust?,"Other Insect? (catipillar bites, etc.)","Mechanical? (human, animal, herbicide, etc.)",Other? (Please mark),Date Labeled,Time Start,Time End
0,NaN,7,28,Exp,NaN,5.0,No,Yes,Yes,Yes,NaN,NaN,NaN,NaN,NaN
1,NaN,7,28,Exp,NaN,12.0,No,No,No,Yes,NaN,NaN,NaN,NaN,NaN
2,NaN,7,28,Exp,NaN,1.0,NaN,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN
3,NaN,7,28,Exp,NaN,10.0,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,NaN
4,NaN,7,28,Exp,NaN,4.0,NaN,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN


### Merge in the `preliminary_labels_df`

In [41]:
preliminary_labels_df.head()

,ID,Site ID,Stop Number,Level Extracted,Date,Time Start,End,Imaging Session,Damage Label,Label Confidence,Notes,Unnamed: 11
0,S1D16-1,1,-,-,1/16/2025,10:30 AM,1:00 PM,2,Rust,Clear,spots,NaN
1,S1D16-10,1,-,-,1/16/2025,10:30 AM,1:00 PM,2,Rust,Clear,NaN,NaN
2,S1D16-11,1,-,-,1/16/2025,10:30 AM,1:00 PM,2,Rust,Clear,NaN,NaN
3,S1D16-12,1,-,-,1/16/2025,10:30 AM,1:00 PM,2,Rust,Maybe,"could just be damage, some minor spotting",NaN
4,S1D16-13,1,-,-,1/16/2025,10:30 AM,1:00 PM,2,Healthy,Small,Tiny spot of rust,NaN


In [42]:
print(preliminary_labels_df["Damage Label"].value_counts())
print(preliminary_labels_df["Stop Number"].value_counts())
print(preliminary_labels_df["Level Extracted"].value_counts())
print(preliminary_labels_df["Site ID"].value_counts())
print(preliminary_labels_df["Date"].value_counts())

Damage Label
Rust             40
Leaf Miner       36
Healthy          21
Wasp-Ant          8
Insect            6
Mechanical        5
-                 3
Insect Damage     1
Name: count, dtype: int64
Stop Number
-    160
1     38
2     33
Name: count, dtype: int64
Level Extracted
-       160
Mid      33
Low      28
High     10
Name: count, dtype: int64
Site ID
1    87
5    41
3    32
7    32
6    27
2    11
4     1
Name: count, dtype: int64
Date
1/21/2025    100
1/19/2025     72
1/16/2025     59
Name: count, dtype: int64


In [43]:
from collections import defaultdict

# Prelim Columns for reference
# ID, Site ID, Stop Number, Level Extracted, Date, Time Start, End,
# Imaging Session, Damage Label, Label Confidence, Notes

# Target Columns for reference
# Leaf ID, Site, Day, Plant, Level, Index, Healthy, Leaf Miner?, Rust?,
# Other Insect? (catipillar bites, etc.), Mechanical? (human, animal, herbicide, etc.),
# Other? (Please mark), Date Labeled, Time Start, Time End

# Possible Labels from preliminary
# Rust             40
# Leaf Miner       36
# Healthy          21
# Wasp-Ant          8
# Insect            6
# Mechanical        5
# -                 3
# Insect Damage

new_df_rows = []
skip_reasons = defaultdict(int)
for i, row in preliminary_labels_df.iterrows():
    leaf_id = None
    site = row["Site ID"]
    month, day, year = row.Date.split("/")
    day = int(day)
    plant = row["Stop Number"]
    plant = None if plant == "-" else int(plant)
    level = row["Level Extracted"]
    level = None if level == "-" else int(plant)
    id_base, index = row.ID.split("-")
    site_base, id_day = id_base.split("D")
    id_site = site_base.split("S")[-1]
    index = int(index)
    assert int(id_site) == site, "Site ids must match"
    assert int(id_day) == day, "Site days must match"
    healthy = None
    leaf_miner = None
    rust = None
    other_insect = None
    mechanical = None
    other = None
    date_labeled = None  # Leaving as None as unsure whether the date and time in the preliminary maches the extraction time or the labeling time
    time_start = None  # Leaving as None as unsure whether the date and time in the preliminary maches the extraction time or the labeling time
    time_end = None  # Leaving as None as unsure whether the date and time in the preliminary maches the extraction time or the labeling time

    prelim_label = row["Damage Label"]
    if pd.isna(prelim_label):
        skip_reasons["Nan label"] += 1
        continue  # We skip these

    if prelim_label in ["Insect", "Insect Damage"]:
        other_insect = "Yes"
    elif prelim_label == "Rust":
        rust = "Yes"
    elif prelim_label == "Healthy":
        healthy = "Yes"
    elif prelim_label == "Leaf Miner":
        leaf_miner = "Yes"
    elif prelim_label == "Mechanical":
        mechanical = "Yes"
    elif prelim_label == "Wasp-Ant":
        other = "Wasp-Ant"  # Not putting into other insect since they don't leave clear bite / eating marks.
    elif prelim_label == "-":
        skip_reasons["Invalid label"] += 1
        continue  # Discard these rows
    else:
        raise NotImplementedError(
            f"{prelim_label} was an unexpected case. Please update accordingly."
        )

    # Leaf ID, Site, Day, Plant, Level, Index, Healthy, Leaf Miner?, Rust?,
    # Other Insect? (catipillar bites, etc.), Mechanical? (human, animal, herbicide, etc.),
    # Other? (Please mark), Date Labeled, Time Start, Time End
    new_df_rows.append(
        [
            leaf_id,
            site,
            day,
            plant,
            level,
            index,
            healthy,
            leaf_miner,
            rust,
            other_insect,
            mechanical,
            other,
            date_labeled,
            time_start,
            time_end,
        ]
    )

translated_preliminary_labels_df = pd.DataFrame(new_df_rows, columns=labels_df.columns)
print(
    f"{preliminary_labels_df.shape[0] - translated_preliminary_labels_df.shape[0]} rows dropped"
)
print(f"{translated_preliminary_labels_df.shape[0]} rows kept")
print([f"{k}:{v}" for k, v in skip_reasons.items()])

labels_df = pd.concat([labels_df, translated_preliminary_labels_df])
labels_df.head()

114 rows dropped
117 rows kept
['Invalid label:3', 'Nan label:111']


,Leaf ID,Site,Day,Plant,Level,Index,Healthy,Leaf Miner?,Rust?,"Other Insect? (catipillar bites, etc.)","Mechanical? (human, animal, herbicide, etc.)",Other? (Please mark),Date Labeled,Time Start,Time End
0,NaN,7,28,Exp,NaN,5.0,No,Yes,Yes,Yes,NaN,NaN,NaN,NaN,NaN
1,NaN,7,28,Exp,NaN,12.0,No,No,No,Yes,NaN,NaN,NaN,NaN,NaN
2,NaN,7,28,Exp,NaN,1.0,NaN,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN
3,NaN,7,28,Exp,NaN,10.0,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,NaN
4,NaN,7,28,Exp,NaN,4.0,NaN,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN


## Linking

### Cleaning label linking

We should be able to merge on `site`, `day`, `plant`, `level`, and `index`. Some of these will be NaN or an invalid value, we can set these to `NA`.

So our key will look like `site-day-plant-level-index`.

`site`, `day`, and `index` should be valid, but `plant` and `level` may have values missing.

In [44]:
print(labels_df.Plant.unique())
print(labels_df.Level.unique())

['Exp' 'P1' 'P2' 'P3' '1' '2' 'exp' 'Exp-P1' 'exp-p2' None]
[nan 'M' 'H' 'L' None]


In [45]:
labels_df_clean = labels_df.copy()

# Set nans to 'NA' in Level
labels_df_clean.Level = labels_df.Level.fillna("NA")
print(labels_df_clean.Level.unique())

# Set nans to 'NA' in Plant
labels_df_clean.Plant = labels_df_clean.Plant.fillna("NA")
# Set 'Exp' and 'exp' to 'NA'
labels_df_clean.Plant = labels_df_clean.Plant.replace(["exp", "Exp"], "NA")
print(labels_df_clean.Plant.unique())

# Set 'P#' and '#' to # and 'Exp-P1' to 1 and 'exp-p2' to 2
labels_df_clean.Plant = labels_df_clean.Plant.replace(["P1", "1", "Exp-P1"], 1)
labels_df_clean.Plant = labels_df_clean.Plant.replace(["P2", "2", "exp-p2"], 2)
labels_df_clean.Plant = labels_df_clean.Plant.replace(["P3", "exp-p2"], 3)
print(labels_df_clean.Plant.unique())
print(labels_df_clean.Index.unique())

# Drop nan for index since we don't know if that was an accidental new row in the labeling sheet
labels_df_clean = labels_df_clean[labels_df_clean.Index.notna()]
labels_df_clean.Index = labels_df_clean.Index.astype(int)
print(labels_df_clean.Index.unique())

['NA' 'M' 'H' 'L']
['NA' 'P1' 'P2' 'P3' '1' '2' 'Exp-P1' 'exp-p2']
['NA' 1 2 3]
[ 5. 12.  1. 10.  4.  9. 14.  8.  2.  3.  6. 13.  7. 11. 18. 19. 27. 21.
 24. 23. 26. 25. 15. 22. 17. 20. 16. 28. 29. 30. nan 31. 32. 33. 34. 35.
 36. 37. 38. 39. 40. 41. -2. -1. 42. 43. 44. 45. 46. 47. 48. 49. 50. 51.
 52. 53. 54. 55. 56. 57. 58. 59. 60. 61. 62. 63. 64. 65. 66. 67. 68. 69.
 70. 71. 72. 73.]
[ 5 12  1 10  4  9 14  8  2  3  6 13  7 11 18 19 27 21 24 23 26 25 15 22
 17 20 16 28 29 30 31 32 33 34 35 36 37 38 39 40 41 -2 -1 42 43 44 45 46
 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70
 71 72 73]


In [46]:
# Handling duplicates
print(labels_df_clean["Index"].value_counts())
print(
    labels_df_clean.duplicated(subset=["Site", "Day", "Plant", "Level", "Index"]).sum()
)
labels_df_clean = labels_df_clean.drop_duplicates(
    subset=["Site", "Day", "Plant", "Level", "Index"]
)
print(
    labels_df_clean.duplicated(subset=["Site", "Day", "Plant", "Level", "Index"]).sum()
)
labels_df_clean.head()


Index
2     54
4     53
1     52
3     52
5     51
      ..
69     1
70     1
71     1
72     1
73     1
Name: count, Length: 75, dtype: int64
79
0


,Leaf ID,Site,Day,Plant,Level,Index,Healthy,Leaf Miner?,Rust?,"Other Insect? (catipillar bites, etc.)","Mechanical? (human, animal, herbicide, etc.)",Other? (Please mark),Date Labeled,Time Start,Time End
0,NaN,7,28,NA,NA,5,No,Yes,Yes,Yes,NaN,NaN,NaN,NaN,NaN
1,NaN,7,28,NA,NA,12,No,No,No,Yes,NaN,NaN,NaN,NaN,NaN
2,NaN,7,28,NA,NA,1,NaN,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN
3,NaN,7,28,NA,NA,10,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,NaN
4,NaN,7,28,NA,NA,4,NaN,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN


In [47]:
labels_df_clean.Index.unique()

array([ 5, 12,  1, 10,  4,  9, 14,  8,  2,  3,  6, 13,  7, 11, 18, 19, 27,
       21, 24, 23, 26, 25, 15, 22, 17, 20, 16, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, -2, -1, 42, 43, 44, 45, 46, 47, 48, 49,
       50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66,
       67, 68, 69, 70, 71, 72, 73])

In [48]:
# Save the cleaned labels
labels_df_clean.to_csv(metadata_path_root / "clean_labels.csv")

### Cleaning Image Metadata labels

In [49]:
clean_image_metadata_df = image_metadata_df.copy()
clean_image_metadata_df = clean_image_metadata_df[
    clean_image_metadata_df["Original\nFilename"].notna()
]  # Drop invalid rows. Use 'Original\nFilename' column

print(clean_image_metadata_df["Plant #"].unique())
print(clean_image_metadata_df.Level.unique())

clean_image_metadata_df["Plant #"] = clean_image_metadata_df["Plant #"].fillna("NA")
clean_image_metadata_df.Level = clean_image_metadata_df.Level.fillna("NA")

print(clean_image_metadata_df["Plant #"].unique())
print(clean_image_metadata_df.Level.unique())

print(
    clean_image_metadata_df.duplicated(
        subset=["Site", "Day", "Plant #", "Level", "Index", "View"]
    ).sum()
)

clean_image_metadata_df = clean_image_metadata_df.drop_duplicates(
    subset=["Site", "Day", "Plant #", "Level", "Index", "View"]
)

print(
    clean_image_metadata_df.duplicated(
        subset=["Site", "Day", "Plant #", "Level", "Index", "View"]
    ).sum()
)

# Save the cleaned image metadata
clean_image_metadata_df.to_csv(metadata_path_root / "clean_image_metadata.csv")

clean_image_metadata_df.head()

[nan  1.]
[nan 'H' 'L' 'M']
['NA' 1.0]
['NA' 'H' 'L' 'M']
4
0


,UUID,Original\nFilename,Site,Day,Plant #,Level,Index,View,Imaging\nSession #,Session\nFolder Name,Name of \nMapper,Recheck\nImage?,Notes
0,NaN,IMG_0861.png,5.0,21.0,NA,NA,1.0,D,4.0,session_4_1_21_2025,David,False,NaN
1,NaN,IMG_0862.png,5.0,21.0,NA,NA,1.0,V,4.0,session_4_1_21_2025,David,False,NaN
2,NaN,IMG_0865.png,5.0,21.0,NA,NA,2.0,D,4.0,session_4_1_21_2025,David,False,NaN
3,NaN,IMG_0866.png,5.0,21.0,NA,NA,2.0,V,4.0,session_4_1_21_2025,David,False,NaN
4,NaN,IMG_0871.png,5.0,21.0,NA,NA,3.0,D,4.0,session_4_1_21_2025,David,False,Seem to be missing ventral view from dataset


### Merging the data

In [50]:
# Load the data
labels_df_clean = pd.read_csv(metadata_path_root / "clean_labels.csv")
clean_image_metadata_df = pd.read_csv(metadata_path_root / "clean_image_metadata.csv")

labels_df_clean = labels_df_clean.fillna("NA")

label_link_key = (
    labels_df_clean.Site.astype(str)
    + "_"
    + labels_df_clean.Day.astype(str)
    + "_"
    + labels_df_clean.Plant.astype(str)
    + "_"
    + labels_df_clean.Level.astype(str)
    + "_"
    + labels_df_clean.Index.astype(str)
)
label_link_key = label_link_key.str.replace(
    ".0", ""
)  # Some ints are floats due to nans

clean_image_metadata_df = clean_image_metadata_df.fillna("NA")

image_metadata_link_key = (
    clean_image_metadata_df.Site.astype(str)
    + "_"
    + clean_image_metadata_df.Day.astype(str)
    + "_"
    + clean_image_metadata_df["Plant #"].astype(str)
    + "_"
    + clean_image_metadata_df.Level.astype(str)
    + "_"
    + clean_image_metadata_df.Index.astype(str)
)
image_metadata_link_key = image_metadata_link_key.str.replace(".0", "")

assert not label_link_key.str.contains(".", regex=False).any(), "Can't have decimals"
assert not image_metadata_link_key.str.contains(".", regex=False).any(), (
    "Can't have decimals"
)

labels_df_clean["link_key"] = label_link_key
clean_image_metadata_df["link_key"] = image_metadata_link_key

linked_metadata_df = pd.merge(
    left=clean_image_metadata_df, right=labels_df_clean, how="inner", on="link_key"
)
print(labels_df_clean.shape)
print(clean_image_metadata_df.shape)
print(linked_metadata_df.shape)
linked_metadata_df.to_csv(metadata_path_root / "linked_metadata.csv")


(764, 17)
(521, 15)
(515, 31)


## Checking for duplications

In [51]:
print(labels_df_clean.duplicated("link_key").sum())
print(clean_image_metadata_df.duplicated(["link_key", "View"]).sum())
print(linked_metadata_df.duplicated(["link_key", "View"]).sum())

0
0
0
